# 107 — DS5 Balanced Phenotype Profile Chart

**Purpose:** Re-visualize DS5 phenotype profiles with **one representative feature per model
family**, making the two-layer structure (Layer 1 = NeT, Layer 2 = StoP/DyS/StM/InT)
explicit rather than burying it under a NeT-heavy radar.

**What this is NOT:** A new analysis. All clusters and features come from existing
`outputs/model_outputs/` files produced by notebooks 41–45 and 100–106.
Nothing in those notebooks is modified.

**What this IS:** A visualization-level decision: replace stacked NeT metrics with a
balanced 5-axis radar so all model families are represented equally in the figure.

**Context:** See `outputs/reports/planning/phenotyping_improvement_proposals.md` for the
full three-level improvement plan. This notebook implements Level 1 only.

---

**Feature choices (one per model family):**

| Axis | Feature | Model family | Why |
|---|---|---|---|
| NeT | `HHI_net` | Network Theory | Hub concentration — primary split axis |
| StoP | `sig_hat` | Stochastic Processes | Noise / stochasticity level |
| InT | `H_emp` | Information Theory | Empirical entropy of override behavior |
| DyS | `deviation_from_predicted_fixed_point` | Dynamical Systems | Distance from predicted resting state |
| StM | `chi_behav` | Statistical Mechanics | Susceptibility (tipping-point proximity); not redundant with `sig_hat` unlike `beta_behav` |

**Two plots produced:**
1. **Phase 1 radar** — Profile A vs Profile B (the main fig4 target)
2. **Layer 2 radar** — C1/C2/C3 from Path A (behavior-only, NeT excluded) — shows that
   behavioral dynamics DO differentiate participants once NeT is set aside

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

ROOT = Path("..")
MODEL_OUT = ROOT / "outputs" / "model_outputs"
PHENOTYPE_OUT = MODEL_OUT / "phenotyping"
FIG_OUT = PHENOTYPE_OUT / "figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)

print("Paths OK")
print(f"Phenotyping outputs: {PHENOTYPE_OUT}")

In [ ]:
# --- Load Phase 1 cluster assignments ---
phase1 = pd.read_csv(PHENOTYPE_OUT / "ds5_cluster_assignments.csv")[
    ["participant_id_canonical", "profile_p1"]
]
print(f"Phase 1 assignments: {len(phase1)} participants")
print(phase1["profile_p1"].value_counts())

In [ ]:
# --- Load one feature per model family ---
net = pd.read_csv(MODEL_OUT / "network_models" / "participant_network_metrics.csv")[
    ["participant_id_canonical", "HHI_net"]
]
stop = pd.read_csv(MODEL_OUT / "stochastic_processes" / "participant_parameters.csv")[
    ["participant_id_canonical", "sig_hat"]
]
int_ = pd.read_csv(MODEL_OUT / "information_theory" / "participant_entropy_metrics.csv")[
    ["participant_id_canonical", "H_emp"]
]
dys = pd.read_csv(MODEL_OUT / "dynamical_systems" / "participant_dynamics_parameters.csv")[
    ["participant_id_canonical", "deviation_from_predicted_fixed_point"]
]
stm = pd.read_csv(MODEL_OUT / "statistical_mechanics" / "participant_stm_parameters.csv")[
    ["participant_id_canonical", "chi_behav"]
]

print(f"NeT: {len(net)}, StoP: {len(stop)}, InT: {len(int_)}, DyS: {len(dys)}, StM: {len(stm)}")

In [ ]:
# --- Merge all features with Phase 1 cluster assignment ---
# Use inner joins: participant must have Phase 1 assignment AND all 5 model features
df = (
    phase1
    .merge(net, on="participant_id_canonical", how="inner")
    .merge(stop, on="participant_id_canonical", how="inner")
    .merge(int_, on="participant_id_canonical", how="inner")
    .merge(dys, on="participant_id_canonical", how="inner")
    .merge(stm, on="participant_id_canonical", how="inner")
)

FEATURES = ["HHI_net", "sig_hat", "H_emp", "deviation_from_predicted_fixed_point", "chi_behav"]
FEATURE_LABELS = [
    "HHI_net\n(NeT)",
    "σ̂ noise\n(StoP)",
    "H_emp entropy\n(InT)",
    "Δ fixed-point\n(DyS)",
    "χ susceptibility\n(StM)",
]

print(f"Merged: {len(df)} participants with all features")
print(f"Profile distribution: {df['profile_p1'].value_counts().to_dict()}")
print()
print("Raw per-profile means:")
print(df.groupby("profile_p1")[FEATURES].mean().round(4))

In [ ]:
# --- Normalize each feature to [0, 1] using full-sample min/max ---
# So 0 = lowest observed in DS5, 1 = highest observed in DS5
df_norm = df.copy()
for feat in FEATURES:
    fmin, fmax = df[feat].min(), df[feat].max()
    df_norm[feat] = (df[feat] - fmin) / (fmax - fmin)

# Per-profile means on normalized features
profile_means_norm = df_norm.groupby("profile_p1")[FEATURES].mean()

print("Normalized per-profile means (0=sample min, 1=sample max):")
print(profile_means_norm.round(3))

In [ ]:
def plot_balanced_radar(
    profile_means,
    feature_labels,
    profile_colors,
    profile_labels,
    title="",
    figsize=(7, 6),
    ylim=(0, 1),
):
    """
    Balanced radar chart: one feature per model family.
    profile_means: dict of {label: array of normalized values per feature}
    """
    n = len(feature_labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"polar": True})

    for label, color in zip(profile_labels, profile_colors):
        values = profile_means[label].tolist()
        values += values[:1]  # close
        ax.plot(angles, values, color=color, linewidth=2.2, linestyle="solid", label=label)
        ax.fill(angles, values, color=color, alpha=0.15)

    # Axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(feature_labels, size=10)
    ax.set_ylim(*ylim)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.0"], size=7, color="grey")
    ax.yaxis.set_tick_params(labelsize=7)
    ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.5)

    if title:
        ax.set_title(title, size=12, pad=18, fontweight="bold")

    ax.legend(
        loc="upper right",
        bbox_to_anchor=(1.35, 1.15),
        frameon=True,
        fontsize=10,
    )

    return fig, ax

## Plot 1 — Phase 1: Profile A vs Profile B (balanced radar)

This is the improved `fig4` target. One axis per model family.

**What to expect:** NeT axis (HHI_net) should show a very large gap between A and B.
All other axes should show little separation — this is the honest picture: Layer 1
(network topology) strongly separates the profiles; Layer 2 features (StoP/InT/DyS/StM)
are similar between A and B. The chart makes that explicit.

In [ ]:
# Build per-profile value arrays (in feature order)
p1_values = {}
for profile in ["Profile A", "Profile B"]:
    p1_values[profile] = profile_means_norm.loc[profile, FEATURES].values

COLORS_P1 = ["#2196F3", "#FF5722"]  # blue = A, orange-red = B
PROFILES_P1 = ["Profile A", "Profile B"]

fig1, ax1 = plot_balanced_radar(
    p1_values,
    FEATURE_LABELS,
    profile_colors=COLORS_P1,
    profile_labels=PROFILES_P1,
    title="DS5 Phase 1 — Balanced Profile Radar\n(one feature per model family)",
)

# Annotation: note the two-layer interpretation
fig1.text(
    0.5, -0.04,
    "Profile A (N=125, distributed network) vs Profile B (N=289, hub-locked network)\n"
    "Normalized to sample range [0=min, 1=max]. NeT axis dominates the split (NeT-only ARI vs Phase 1 = 0.990).",
    ha="center", va="top", fontsize=8, color="#555555",
    transform=fig1.transFigure,
)

plt.tight_layout()
fig1.savefig(FIG_OUT / "fig4v2_balanced_phase1_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig4v2_balanced_phase1_radar.png")

## Plot 2 — Layer 2: C1 / C2 / C3 from Path A (behavior-only clusters)

Path A excluded NeT features entirely and found k=3 (C1/C2/C3) with ARI vs Phase 1 = 0.020
(orthogonal). This radar shows that when NeT is excluded, the remaining 4 axes **do**
differentiate participants — this is the behavioral dynamics layer that is invisible in
the NeT-dominated Phase 1 chart.

Note: HHI_net is still included on the axis but C1/C2/C3 are NOT separated by it
(by construction). This confirms the orthogonality visually.

In [ ]:
# Load Path A (behavior-only) cluster assignments
patha = pd.read_csv(PHENOTYPE_OUT / "ds5_behav_only_cluster_assignments.csv")
print("Path A columns:", patha.columns.tolist())
print(patha.iloc[:2])

In [ ]:
# cluster_behav = Path A's k=3 solution (C1/C2/C3, behavior-only, NeT excluded from clustering)
# Column confirmed from ds5_behav_only_cluster_assignments.csv header
behav_label_col = "cluster_behav"
print(f"Using column: {behav_label_col}")

patha_slim = patha[["participant_id_canonical", behav_label_col]].rename(
    columns={behav_label_col: "cluster_patha"}
)
print(patha_slim["cluster_patha"].value_counts())

In [ ]:
df_patha = (
    patha_slim
    .merge(net, on="participant_id_canonical", how="inner")
    .merge(stop, on="participant_id_canonical", how="inner")
    .merge(int_, on="participant_id_canonical", how="inner")
    .merge(dys, on="participant_id_canonical", how="inner")
    .merge(stm, on="participant_id_canonical", how="inner")
)

# Normalize using the same full-sample range as Phase 1 (for comparability)
df_patha_norm = df_patha.copy()
for feat in FEATURES:
    fmin, fmax = df[feat].min(), df[feat].max()  # use Phase 1 merged sample range
    df_patha_norm[feat] = (df_patha[feat] - fmin) / (fmax - fmin)

cluster_means_norm = df_patha_norm.groupby("cluster_patha")[FEATURES].mean()
print("Normalized per-cluster means (C1/C2/C3):")
print(cluster_means_norm.round(3))

In [ ]:
# Plot Layer 2 (Path A clusters)
cluster_labels_patha = cluster_means_norm.index.tolist()
patha_values = {label: cluster_means_norm.loc[label, FEATURES].values for label in cluster_labels_patha}

COLORS_L2 = ["#4CAF50", "#9C27B0", "#FF9800"]  # green / purple / amber

fig2, ax2 = plot_balanced_radar(
    patha_values,
    FEATURE_LABELS,
    profile_colors=COLORS_L2,
    profile_labels=cluster_labels_patha,
    title="DS5 Layer 2 (Path A) — Behavior-Only Clusters\n(same axes, NeT excluded from clustering)",
)

fig2.text(
    0.5, -0.04,
    "C1=low-noise/low-override, C2=low-noise/high-override, C3=high-noise/variable\n"
    "ARI(Layer 2 vs Phase 1) = 0.020 — orthogonal to network-topology split.\n"
    "Same normalization as Plot 1 (sample range from Phase 1 population).",
    ha="center", va="top", fontsize=8, color="#555555",
    transform=fig2.transFigure,
)

plt.tight_layout()
fig2.savefig(FIG_OUT / "fig4v2_balanced_layer2_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig4v2_balanced_layer2_radar.png")

## Combined two-panel figure (Layer 1 + Layer 2)

Side-by-side: the two independent phenotype layers on the same balanced axes.
This is the thesis-ready version that shows the orthogonality visually.

In [ ]:
fig3, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"polar": True})

n = len(FEATURE_LABELS)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

def _draw_on_ax(ax, values_dict, colors, labels, title):
    for label, color in zip(labels, colors):
        vals = values_dict[label].tolist() + [values_dict[label][0]]
        ax.plot(angles, vals, color=color, linewidth=2.2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.13)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(FEATURE_LABELS, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], size=6, color="grey")
    ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.4)
    ax.set_title(title, size=11, pad=20, fontweight="bold")
    ax.legend(loc="upper right", bbox_to_anchor=(1.42, 1.18), fontsize=9, frameon=True)

_draw_on_ax(
    axes[0], p1_values, COLORS_P1, PROFILES_P1,
    "Layer 1 — Network topology\n(Phase 1, k=2, ARI NeT-only=0.990)"
)
_draw_on_ax(
    axes[1], patha_values, COLORS_L2, cluster_labels_patha,
    "Layer 2 — Behavioral dynamics\n(Path A, k=3, ARI vs Layer1=0.020)"
)

fig3.suptitle(
    "DS5 — Two Independent Phenotype Layers (balanced 5-family radar)",
    fontsize=13, fontweight="bold", y=1.02
)
fig3.text(
    0.5, -0.02,
    "Each axis = one CAS model family representative feature, normalized to sample range [0, 1]."
    " Left panel: Layer 1 (NeT-driven, condition-stable)."
    " Right panel: Layer 2 (StoP/DyS/StM-driven, condition-modulated).",
    ha="center", va="top", fontsize=8.5, color="#444444",
    transform=fig3.transFigure
)

plt.tight_layout()
fig3.savefig(FIG_OUT / "fig4v2_two_layer_balanced_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig4v2_two_layer_balanced_radar.png")

## Summary

**Three figures saved to `outputs/model_outputs/phenotyping/figures/`:**
- `fig4v2_balanced_phase1_radar.png` — Phase 1 Profile A vs B (improved fig4)
- `fig4v2_balanced_layer2_radar.png` — Layer 2 C1/C2/C3 (Path A behavior-only)
- `fig4v2_two_layer_balanced_radar.png` — Combined two-panel (thesis-ready)

**What to observe:**
- Left panel (Layer 1): NeT axis dominates — honest representation of what Phase 1 finds
- Right panel (Layer 2): non-NeT axes differentiate — C1/C2/C3 visible on StoP/DyS/StM/InT
- Together: two orthogonal structures, both present in the data, visible on the same axes

**Next steps (from `outputs/reports/planning/phenotyping_improvement_proposals.md`):**
- Level 2: remove NeT self-loops (DS5_02, A3) — may soften the Layer 1 NeT axis
- Level 3: V(x)/Boltzmann reformulation + GMM re-cluster (DS5_02, Phase A→B) — may make
  Layer 2 axes richer and more differentiated even before NeT revision

**Nothing in notebooks 100–106 or 41–45 was modified.**